# ML Model Training on Backend Logs

This notebook loads the `activity_logs` table from the backend SQLite database, trains a machine learning model using those records, performs cross-validation, and evaluates QR scanning and OCR accuracy metrics.

## 1. Import Required Libraries

Import necessary libraries including pandas, scikit-learn, numpy, matplotlib, and any other auxiliary packages.

In [ ]:
import sqlite3
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, cross_val_score, KFold
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.metrics import classification_report, confusion_matrix, mean_squared_error, r2_score

%matplotlib inline

## 2. Load and Explore Backend Data

Load the backend dataset from SQLite, display basic statistics, check for missing values, and visualise distributions.

In [ ]:
# connect to sqlite database
conn = sqlite3.connect('apsrtc_local.db')

df = pd.read_sql_query('SELECT * FROM activity_logs', conn)

df.head()

In [ ]:
# summary info
print(df.info())
print(df.isnull().sum())

# stats
print(df.describe())

# visualize distributions
sns.histplot(df['response_time'].dropna(), kde=True); plt.title('Response Time');
sns.histplot(df['ocr_accuracy'].dropna(), kde=True); plt.title('OCR Accuracy');

## 3. Data Preprocessing and Feature Engineering

Clean the data, handle missing values, encode categorical variables, and create relevant features for model training.

In [ ]:
# basic cleaning
clean = df.copy()
clean = clean.dropna(subset=['status_code','response_time'])

# fill missing OCR
clean['ocr_accuracy'] = clean['ocr_accuracy'].fillna(clean['ocr_accuracy'].median())

# encode categorical
clean['event_type_enc'] = clean['event_type'].astype('category').cat.codes
if 'scan_method' in clean.columns:
    clean['scan_method_enc'] = clean['scan_method'].astype('category').cat.codes
else:
    clean['scan_method_enc'] = 0

# create target variables
if 'anomaly' not in clean.columns:
    clean['anomaly'] = (clean['status_code'] >= 400).astype(int)

features = ['status_code','response_time','ocr_accuracy','event_type_enc','scan_method_enc']
X = clean[features]
y_class = clean['anomaly']
y_reg = clean['ocr_accuracy']

X.head(), y_class.value_counts()

## 4. Train ML Model

Split data into training and testing sets, and train classifiers/regressors on relevant targets.

In [ ]:
# split data
X_train, X_test, y_class_train, y_class_test = train_test_split(X, y_class, test_size=0.3, random_state=42)
X_train_reg, X_test_reg, y_reg_train, y_reg_test = train_test_split(X, y_reg, test_size=0.3, random_state=42)

# classifier for anomaly detection
clf = RandomForestClassifier(n_estimators=100, random_state=42)
clf.fit(X_train, y_class_train)

# regressor for OCR accuracy
reg = RandomForestRegressor(n_estimators=100, random_state=42)
reg.fit(X_train_reg, y_reg_train)


## 5. Model Cross-Validation

Perform k-fold cross-validation to evaluate model performance with accuracy and error metrics.

In [ ]:
kf = KFold(n_splits=5, shuffle=True, random_state=42)

scores_clf = cross_val_score(clf, X, y_class, cv=kf, scoring='accuracy')
print('Classifier CV accuracy:', scores_clf, scores_clf.mean())

scores_reg = cross_val_score(reg, X, y_reg, cv=kf, scoring='r2')
print('Regressor CV R2:', scores_reg, scores_reg.mean())

## 6. Make Predictions on Test Data

Use the trained models to make predictions and compare with actual values.

In [ ]:
# classifier predictions
pred_class = clf.predict(X_test)
print(classification_report(y_class_test, pred_class))

# regressor predictions
pred_reg = reg.predict(X_test_reg)
print('MSE', mean_squared_error(y_reg_test, pred_reg))
print('R2', r2_score(y_reg_test, pred_reg))

## 7. Evaluate QR Scanning Accuracy

Test QR code scanning accuracy on sample data, calculate success rate, and identify failure cases.

In [ ]:
# placeholder: QR scanning data not currently in logs
# if you have ground truth labels for scans, load and calculate accuracy
# e.g.,
# qr_df = pd.read_sql_query('SELECT qr_data, success FROM qr_logs', conn)
# print('QR accuracy', qr_df['success'].mean())

print('QR evaluation requires separate dataset.')

## 8. Evaluate OCR Accuracy

Evaluate Optical Character Recognition accuracy by comparing extracted text with ground truth labels and calculating error metrics.

In [ ]:
# assume we have some ground truth OCR records with 'true_text' and 'predicted_text'
# ocr_df = pd.read_sql_query('SELECT * FROM ocr_logs', conn)
# ocr_df['accuracy'] = ocr_df.apply(lambda r: 1 - editdistance.eval(r.true_text, r.predicted_text)/max(len(r.true_text),1), axis=1)
# print('Mean OCR accuracy', ocr_df['accuracy'].mean())

print('Add OCR dataset to perform this evaluation.')

## 9. Compare API Service Metrics

Compile model performance, QR scan accuracy (if available), and OCR accuracy into a unified report.